## shouldSplit: train + inference

Ниже короткий рабочий блок для:
1. обучения `shouldSplit` модели через существующий pipeline,
2. сохранения артефакта,
3. инференса по новым объявлениям из артефакта.

In [1]:
import avito

In [1]:
import joblib
import pandas as pd


from common.paths import PathLike, get_avito_data_dpath
from avito.config import AvitoCaseConfig
from avito.embeddings import EncoderConfig, SentenceTransformerEncoder
from avito.should_split.classifier import train_should_split_models
from avito.should_split.inference import predict_should_split_from_artifact

data_fpath = get_avito_data_dpath() / "rnc_dataset_markup.json"

data_df = pd.read_json(data_fpath)
data_df.head()

,itemId,sourceMcId,sourceMcTitle,description,targetDetectedMcIds,targetSplitMcIds,shouldSplit,caseType,split
0,1000001,101,Ремонт квартир и домов под ключ,"Всё виды строительных работ\r\nКачественно, в ...",[],[],False,no_other_microcategories_detected,NaN
1,1000002,101,Ремонт квартир и домов под ключ,Профессионально и качественно сделаем ремонт к...,"[102, 103, 105, 108, 109, 110]",[],False,other_microcategories_detected_but_not_split,NaN
2,1000003,101,Ремонт квартир и домов под ключ,"ремонт квартир, ванной комнате , балкон",[102],[],False,other_microcategories_detected_but_not_split,NaN
3,1000004,101,Ремонт квартир и домов под ключ,ЗBОНИТЕ KОHСУЛЬТАЦИЯ БЕCПЛАTНAЯ ПO ТУЛЬСKOЙ ОБ...,"[102, 103, 104, 105, 106, 107, 108, 109, 110]","[102, 103, 104, 105, 106, 107, 108, 109, 110]",True,split,NaN
4,1000005,101,Ремонт квартир и домов под ключ,Ремонт квартир любой сложности. Квартиры под к...,[],[],False,no_other_microcategories_detected,NaN


In [2]:
encoder = SentenceTransformerEncoder(EncoderConfig.from_default_yaml())

def encode_texts(df: pd.DataFrame) -> list[list[float]]:
    texts = []
    for row in df.itertuples():
        text = f"""
        Description: {row.description}
        SourceMcTitle: {row.sourceMcTitle}
        """
        texts.append(text)
    encoded_texts = encoder.encode(texts)
    return encoded_texts

data_df["encoded_text"] = encode_texts(data_df)

2026-04-07 21:48:11,200 - avito-embeddings - INFO - [AVITO/EMBEDDINGS] Пробую загрузить локальную модель: C:\Users\User\Desktop\dirs\Dev\hack-mfti\avito\checkpoints\rubert-mini-frida
Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.
2026-04-07 21:48:11,465 - avito-embeddings - INFO - [AVITO/EMBEDDINGS] Запуск encode для 2480 текстов


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

In [5]:
from avito.features import extract_should_split_features

data_df = extract_should_split_features(data_df)
data_df.head()

,description_word_count,description_char_count,split_marker_count,complex_marker_count,marker_ratio,has_bullets,source_mc_id,is_turnkey,case_type,sentence_count,avg_word_len,punctuation_ratio,max_keyphrase_rapidfuzz,split_marker_near_keyphrase
0,13.0,85.0,0.0,0.0,0.0,0,101,1,no_other_microcategories_detected,2.0,5.384615,0.023529,29.565218,0
1,107.0,803.0,0.0,1.0,0.0,0,101,1,other_microcategories_detected_but_not_split,18.0,6.222222,0.029888,48.780487,0
2,6.0,39.0,0.0,0.0,0.0,0,101,1,other_microcategories_detected_but_not_split,1.0,6.400000,0.051282,45.714287,0
3,169.0,1247.0,0.0,0.0,0.0,0,101,1,split,10.0,6.193750,0.034483,6.250000,0
4,26.0,181.0,0.0,1.0,0.0,0,101,1,no_other_microcategories_detected,6.0,5.692307,0.044199,89.285713,0
